## Imports

In [1]:
# REINFORCE
# RL boosted decoder

In [2]:
import pickle
import random
from typing import List, Tuple
import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from maze import *

## Setup

In [3]:
def load_samples(pkl_path: str, n_samples: int = 2000, seed: int = 0):
    with open(pkl_path, 'rb') as f:
        all_data = pickle.load(f)
    random.Random(seed).shuffle(all_data)
    picked = all_data[:n_samples]
    mazes = []
    lengths = []

    for maze, length in picked:
        arr = np.array(maze, dtype=np.float32)

        H, W = arr.shape
        newH = H + 1
        newW = W + 1
        padded = np.zeros((newH, newW), dtype=np.float32)
        padded[:H, :W] = arr

        mazes.append(padded)
        lengths.append(float(length))

    return mazes, lengths


In [4]:
class MazeDataset(Dataset):
    def __init__(self, mazes: List[np.ndarray], lengths: List[float]):
        assert len(mazes) == len(lengths)
        self.mazes = mazes
        self.lengths = lengths
        self.n = len(mazes)
        # assume all mazes same shape
        self.h, self.w = mazes[0].shape


    def __len__(self):
        return self.n


    def __getitem__(self, idx):
        maze = self.mazes[idx]
        length = self.lengths[idx]
        # normalize length to float32 and scale roughly to [0,1]
        return torch.from_numpy(maze).unsqueeze(0), torch.tensor(length, dtype=torch.float32)

## Conditional Variational Autoencoder

In [5]:
F = torch.nn.functional

class ConvEncoder(nn.Module):
    def __init__(self, latent_dim=64, in_size=(20,20)):
        super().__init__()
        # assume input H,W divisible by 4 (20x20 common). adjust if needed.
        self.conv1 = nn.Conv2d(1, 32, 4, 2, 1)   # H/2
        self.conv2 = nn.Conv2d(32, 64, 4, 2, 1)  # H/4
        self.conv3 = nn.Conv2d(64, 128, 3, 1, 1)
        H, W = in_size
        feat_h = H // 4
        feat_w = W // 4
        
        self._flat = 128 * feat_h * feat_w
        self.fc_mu = nn.Linear(self._flat, latent_dim)
        self.fc_logvar = nn.Linear(self._flat, latent_dim)
        self.act = nn.GELU()

    def forward(self, x):
        x = self.act(self.conv1(x))
        x = self.act(self.conv2(x))
        x = self.act(self.conv3(x))
        x = x.view(x.size(0), -1)
        mu = self.fc_mu(x)
        logvar = self.fc_logvar(x)
        return mu, logvar

class ConvDecoder(nn.Module):
    def __init__(self, latent_dim=64, len_emb_dim=16, out_size=(20,20)):
        super().__init__()
        H, W = out_size
        feat_h = H // 4
        feat_w = W // 4
        self.fc = nn.Linear(latent_dim + len_emb_dim, 128 * feat_h * feat_w)
        self.act = nn.GELU()
        self.deconv1 = nn.ConvTranspose2d(128, 64, 4, 2, 1)  # *2
        self.deconv2 = nn.ConvTranspose2d(64, 32, 4, 2, 1)   # *2
        self.outc = nn.Conv2d(32, 1, 3, 1, 1)
    def forward(self, zc, H, W):
        h = self.fc(zc)
        feat_h = H // 4
        feat_w = W // 4
        h = h.view(h.size(0), 128, feat_h, feat_w)
        h = self.act(h)
        h = self.act(self.deconv1(h))
        h = self.act(self.deconv2(h))
        logits = self.outc(h)
        return logits


In [6]:
class ConditionalVAE(nn.Module):
    def __init__(self, latent_dim=64, len_emb_dim=16, in_size=(20,20)):
        super().__init__()
        self.enc = ConvEncoder(latent_dim=latent_dim, in_size=in_size)
        self.len_embed = nn.Sequential(nn.Linear(1, len_emb_dim), nn.GELU(), nn.Linear(len_emb_dim, len_emb_dim))
        self.dec = ConvDecoder(latent_dim=latent_dim, len_emb_dim=len_emb_dim, out_size=in_size)
        self.latent_dim = latent_dim
        self.in_size = in_size

    def reparam(self, mu, logvar):
        std = (0.5 * logvar).exp()
        eps = torch.randn_like(std)
        return mu + eps * std

    def forward(self, x, length):
        mu, logvar = self.enc(x)
        z = self.reparam(mu, logvar)
        le = self.len_embed(length.unsqueeze(1))
        zc = torch.cat([z, le], dim=1)
        logits = self.dec(zc, *self.in_size)
        return logits, mu, logvar

    def generate(self, length, n=1, device='cpu'):
        self.to(device)
        self.eval()
        with torch.no_grad():
            z = torch.randn(n, self.latent_dim, device=device)
            le = torch.tensor([length]*n, dtype=torch.float32, device=device).unsqueeze(1)
            le = self.len_embed(le)
            zc = torch.cat([z, le], dim=1)
            logits = self.dec(zc, *self.in_size)
            probs = torch.sigmoid(logits)
            return (probs > 0.5).long().cpu().numpy().squeeze(1)


In [7]:
def kld_loss(mu, logvar):
    return -0.5 * torch.mean(1 + logvar - mu.pow(2) - logvar.exp())

def train_cvae(pkl_path='mazes.pkl', n_samples=2000, batch_size=64, epochs=20, lr=1e-3, latent_dim=64, device='mps'):
    mazes, lengths = load_samples(pkl_path, n_samples=n_samples)
    # infer input size from first sample
    H, W = mazes[0].shape
    ds = MazeDataset(mazes, lengths)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=True)

    model = ConditionalVAE(latent_dim=latent_dim, len_emb_dim=16, in_size=(H,W)).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    bce = nn.BCEWithLogitsLoss()
    beta = 1e-3

    recon_hist = []
    kld_hist = []
    for epoch in range(1, epochs+1):
        model.train()
        running_r = 0.0
        running_k = 0.0
        for xb, lengths in dl:
            xb = xb.to(device)
            lengths = lengths.to(device)
            opt.zero_grad()
            logits, mu, logvar = model(xb, lengths)
            loss_recon = bce(logits, xb)
            loss_kld = kld_loss(mu, logvar)
            loss = loss_recon + beta * loss_kld
            loss.backward()
            opt.step()
            running_r += loss_recon.item()
            running_k += loss_kld.item()
        avg_r = running_r / len(dl)
        avg_k = running_k / len(dl)
        recon_hist.append(avg_r); kld_hist.append(avg_k)
        print(f"Epoch {epoch}: recon={avg_r:.4f} kld={avg_k:.6f} total={(avg_r+beta*avg_k):.4f}")
    # final loss plot
    plt.figure(); plt.plot(recon_hist,label='recon'); plt.plot(kld_hist,label='kld'); plt.legend(); plt.show()
    return model


In [8]:
def f(maze: list[list[int]], start: tuple = (0, 0), end: tuple | None = None) -> int:
    return 1 if path_len(maze, start, end) != -1 else 0

# NEW ADDED FUNCTION AND WRAPPER
def path_len(maze: list[list[int]], start: tuple = (0, 0), end: tuple | None = None) -> int:
    if end is None:
        end = (len(maze) - 1, len(maze[0]) - 1)
    startx, starty = start
    endx, endy = end
    if maze[startx][starty] == 1 or maze[endx][endy] == 1:
        return -1
    r, c = len(maze), len(maze[0])
    visited = [[False for _ in range(c)] for _ in range(r)]
    visited[startx][starty] = True
    queue = deque()
    queue.append((startx, starty, 0))
    directions = [(0, 1), (1, 0), (0, -1), (-1, 0)]
    while queue:
        x, y, dist = queue.popleft()
        if (x, y) == (endx, endy):
            return dist
        for dx, dy in directions:
            nx, ny = x + dx, y + dy
            if 0 <= nx < r and 0 <= ny < c and not visited[nx][ny] and maze[nx][ny] == 0:
                visited[nx][ny] = True
                queue.append((nx, ny, dist + 1))
    return -1

In [9]:
import copy
import torch
import torch.nn.functional as F
import math

mazes, lengths_data = load_samples('dataset/solvable_data.pkl')


def sample_lengths_from_data(lengths, n, device):
    idx = torch.randint(0, len(lengths), (n,), device=device)
    return torch.tensor([lengths[i] for i in idx.tolist()], dtype=torch.float32, device=device)

def gaussian_reward(maze, L, sigma=2.0):
    actual = maze_path_length(maze)
    if actual == -1:
        return 0.0
    diff = actual - L
    return math.exp(-(diff * diff) / (2 * sigma * sigma))
    
def sample_target_lengths(batch_size, device, low=36, high=55):
    return torch.randint(low, high + 1, (batch_size,), device=device).float()

def fine_tune_solvability_reinforce(
    model,
  #  f_solvable,                 # python function: np.array (H,W) {0,1} -> bool/int reward
    lengths_data=lengths_data,
    steps=3000,
    batch_size=1000,
    lr=1e-4,
    device="mps",
    reg_to_old=0.01,            # keep model close to old distribution (stability)
    entropy_bonus=0.001,        # encourage exploration early
):
    model = model.to(device)
    model.train()

    # Freeze a reference copy to prevent collapse (optional but very helpful)
    old = copy.deepcopy(model).to(device).eval()
    for p in old.parameters():
        p.requires_grad_(False)

    opt = torch.optim.Adam(model.parameters(), lr=lr)

    # running baseline to reduce variance
    baseline = 0.0 # running baseline of recent solve rates
    baseline_momentum = 0.95

    H, W = model.in_size # maze dims

    for step in range(1, steps + 1):
        z = torch.randn(batch_size, model.latent_dim, device=device) # sample latent vector from normal distribution
        lengths = sample_target_lengths(batch_size, device)
        #lengths = sample_lengths_from_data(lengths_data, batch_size, device)
        # lengths = torch.zeros(batch_size, device=device) # since we don't care about generating a maze of a certain length i've just fed in zeroes for length.

        le = model.len_embed(lengths.unsqueeze(1)) # length embedding vector
        zc = torch.cat([z, le], dim=1) # concat vectors , decoder input

        logits = model.dec(zc, H, W)
        probs = torch.sigmoid(logits).clamp(1e-6, 1 - 1e-6)

        x = torch.bernoulli(probs)  # (B,1,H,W). convert probabilities (probs) to actual maze cell values
       # x = (probs > 0.5).float()
        
        # ---- reward from solver
        x_np = x.detach().cpu().numpy().astype("int32")[:, 0] # convert to format for maze solver
       # r_list = [1.0 if f_solvable(m) else 0.0 for m in x_np] # list of solvability 'scores' after running dfs solver on each maze in the batch.
        target_lengths = lengths.detach().cpu().numpy()
        r_list = [gaussian_reward(m, t) for m, t in zip(x_np, target_lengths)]
        r = torch.tensor(r_list, dtype=torch.float32, device=device)  # (B,)

        # ---- log probability of sampled mazes under current policy
        # log P(x) = sum_ij x*log p + (1-x)*log(1-p)
        logp = (x * torch.log(probs) + (1 - x) * torch.log(1 - probs)).sum(dim=(1,2,3))  # (B,)

        # ---- baseline + advantage
        r_mean = r.mean().item() # compute average reward across this batch
        baseline = baseline_momentum * baseline + (1 - baseline_momentum) * r_mean # compute running average
        adv = (r - baseline).detach() # better or worse than usual?

        # REINFORCE loss , minimize the negation
        # high advantage, high probability: good maze, model usually generates - good - large val
        # high advantage, low probability: good maze, not often generated
        # negative advantage - do not generate similarly in the future
        
        loss_pg = -(adv * logp).mean() 

        # ---- entropy : encourage exploration, uncertainty
        # H(Bernoulli(p)) = -p log p - (1-p) log(1-p)
        # compute entropy for every pixel across every maze in the batch, sums over all pixels, then averages over the batch
        # ent, how uncertain the model currently is on average
        ent = (-(probs * torch.log(probs) + (1 - probs) * torch.log(1 - probs))).sum(dim=(1,2,3)).mean()
        
        # try to maximize uncertainty within the bounds of the coeff
        loss_ent = -entropy_bonus * ent  # negative, maximize entropy

        with torch.no_grad():
            old_logits = old.dec(zc, H, W)
            old_probs = torch.sigmoid(old_logits).clamp(1e-6, 1 - 1e-6)

        # KL( Bern(p_old) || Bern(p_new) ) per pixel, summed
        # compare old probabilities vs new probabilities to the same vector, if there is a large difference, apply some penalty
        # penalty scaled by reg_to_old to avoid staying too close to original model results
        kl_pix = old_probs * (torch.log(old_probs) - torch.log(probs)) + (1 - old_probs) * (torch.log(1 - old_probs) - torch.log(1 - probs))
        loss_reg = reg_to_old * kl_pix.sum(dim=(1,2,3)).mean()

        # compute loss as addition of advantage, uncertainty, similarity to original
        loss = loss_pg + loss_ent + loss_reg

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()

        if step % 50 == 0:
            solv_list = [1.0 if path_len(m) != -1 else 0.0 for m in x_np]
            solv_rate = float(np.mean(solv_list))
            print(f"step {step:5d}  reward={r_mean:.3f}  solv_rate={solv_rate:.3f}  baseline={baseline:.3f}  loss={loss.item():.3f}")
            #print(f"step {step:5d}  solv_rate={r_mean:.3f}  baseline={baseline:.3f}  loss={loss.item():.3f}")

    return model


In [10]:
loaded_model = torch.load('saved_models/base_model.pt')
loaded_model.eval() 

ConditionalVAE(
  (enc): ConvEncoder(
    (conv1): Conv2d(1, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (conv2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fc_mu): Linear(in_features=3200, out_features=64, bias=True)
    (fc_logvar): Linear(in_features=3200, out_features=64, bias=True)
    (act): GELU(approximate='none')
  )
  (len_embed): Sequential(
    (0): Linear(in_features=1, out_features=16, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=16, out_features=16, bias=True)
  )
  (dec): ConvDecoder(
    (fc): Linear(in_features=80, out_features=3200, bias=True)
    (act): GELU(approximate='none')
    (deconv1): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (deconv2): ConvTranspose2d(64, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (outc): Conv2d(32, 1, kernel_size=(3, 3), stride=(1, 1), pa

In [11]:
model = fine_tune_solvability_reinforce(
    loaded_model,
  #  f_solvable=f,
    steps=3000,
    batch_size=1000,
    lr=1e-4,
    device="mps",
)


step    50  reward=0.203  solv_rate=0.833  baseline=0.181  loss=1.174
step   100  reward=0.202  solv_rate=0.876  baseline=0.202  loss=0.634
step   150  reward=0.215  solv_rate=0.883  baseline=0.211  loss=0.835
step   200  reward=0.210  solv_rate=0.904  baseline=0.213  loss=0.342
step   250  reward=0.203  solv_rate=0.865  baseline=0.216  loss=-0.020
step   300  reward=0.220  solv_rate=0.892  baseline=0.218  loss=0.667
step   350  reward=0.212  solv_rate=0.907  baseline=0.215  loss=0.600
step   400  reward=0.222  solv_rate=0.890  baseline=0.213  loss=0.766
step   450  reward=0.223  solv_rate=0.893  baseline=0.215  loss=0.753
step   500  reward=0.205  solv_rate=0.884  baseline=0.215  loss=0.285
step   550  reward=0.222  solv_rate=0.885  baseline=0.215  loss=0.744
step   600  reward=0.199  solv_rate=0.884  baseline=0.214  loss=-0.109
step   650  reward=0.217  solv_rate=0.881  baseline=0.218  loss=0.572
step   700  reward=0.215  solv_rate=0.870  baseline=0.220  loss=0.327
step   750  reward

In [12]:
import torch
torch.save(model, "solvable_models/B_base_gaussian.pt")

In [14]:
loaded_model = torch.load('saved_models/3k_decoder.pt')
loaded_model.eval() 


ConditionalVAE(
  (enc): ConvEncoder(
    (conv1): Conv2d(1, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (conv2): Conv2d(32, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fc_mu): Linear(in_features=3200, out_features=64, bias=True)
    (fc_logvar): Linear(in_features=3200, out_features=64, bias=True)
    (act): GELU(approximate='none')
  )
  (len_embed): Sequential(
    (0): Linear(in_features=1, out_features=16, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=16, out_features=16, bias=True)
  )
  (dec): ConvDecoder(
    (fc): Linear(in_features=80, out_features=3200, bias=True)
    (act): GELU(approximate='none')
    (deconv1): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (deconv2): ConvTranspose2d(64, 32, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1))
    (outc): Conv2d(32, 1, kernel_size=(3, 3), stride=(1, 1), pa

In [15]:
model = fine_tune_solvability_reinforce(
    loaded_model,
  #  f_solvable=f,
    steps=3000,
    batch_size=1000,
    lr=1e-4,
    device="mps",
)

step    50  reward=0.191  solv_rate=0.830  baseline=0.172  loss=1.046
step   100  reward=0.211  solv_rate=0.853  baseline=0.202  loss=0.347
step   150  reward=0.205  solv_rate=0.881  baseline=0.206  loss=-0.179
step   200  reward=0.219  solv_rate=0.890  baseline=0.208  loss=0.330
step   250  reward=0.200  solv_rate=0.889  baseline=0.208  loss=-0.956
step   300  reward=0.214  solv_rate=0.892  baseline=0.210  loss=0.031
step   350  reward=0.216  solv_rate=0.903  baseline=0.210  loss=0.026
step   400  reward=0.202  solv_rate=0.888  baseline=0.209  loss=-1.083
step   450  reward=0.199  solv_rate=0.875  baseline=0.206  loss=-1.050
step   500  reward=0.216  solv_rate=0.894  baseline=0.208  loss=0.204
step   550  reward=0.196  solv_rate=0.849  baseline=0.203  loss=-1.001
step   600  reward=0.192  solv_rate=0.884  baseline=0.208  loss=-2.134
step   650  reward=0.205  solv_rate=0.865  baseline=0.208  loss=-0.933
step   700  reward=0.195  solv_rate=0.882  baseline=0.210  loss=-1.993
step   750  

In [16]:
import torch
torch.save(model, "solvable_models/B_3k_gaussian.pt")

In [17]:
model = fine_tune_solvability_reinforce(
    model,
  #  f_solvable=f,
    steps=7000,
    batch_size=1000,
    lr=1e-4,
    device="mps",
)

step    50  reward=0.223  solv_rate=0.913  baseline=0.202  loss=1.030
step   100  reward=0.212  solv_rate=0.917  baseline=0.218  loss=-0.742
step   150  reward=0.224  solv_rate=0.943  baseline=0.224  loss=-0.850
step   200  reward=0.217  solv_rate=0.920  baseline=0.220  loss=-0.864
step   250  reward=0.218  solv_rate=0.927  baseline=0.224  loss=-1.319
step   300  reward=0.246  solv_rate=0.935  baseline=0.223  loss=1.386
step   350  reward=0.246  solv_rate=0.941  baseline=0.225  loss=1.539
step   400  reward=0.210  solv_rate=0.916  baseline=0.221  loss=-1.146
step   450  reward=0.236  solv_rate=0.938  baseline=0.224  loss=0.325
step   500  reward=0.219  solv_rate=0.914  baseline=0.224  loss=-1.118
step   550  reward=0.227  solv_rate=0.915  baseline=0.223  loss=-0.245
step   600  reward=0.199  solv_rate=0.907  baseline=0.221  loss=-2.875
step   650  reward=0.230  solv_rate=0.908  baseline=0.222  loss=-0.252
step   700  reward=0.216  solv_rate=0.895  baseline=0.221  loss=-1.478
step   750

In [18]:
torch.save(model, "solvable_models/B_10k_gaussian.pt")